# π0.5 × LIBERO-Plus 追加学習（Colab）

teleop で集めた教師データ（LeRobot 形式で Hugging Face に置いたもの）で **π0.5 を LoRA で追加学習し、
LIBERO-Plus のシミュレータで動かして動画に残す** notebook。
[SmolVLA 版](smolvla_libero_finetune.ipynb) と同じ流れで、学習の条件は parc2026 で π0.5 を学習したときの
Colab notebook（LeRobot v0.6.0）に合わせている。

```text
teleop で操作・記録（Mac） ─▶ LeRobot データセットにして HF へ ─▶ この notebook
                                                                  ├ 追加学習（Action Expert に LoRA）
                                                                  ├ 学習済みモデルをまとめる（LoRA をマージ）
                                                                  ├ HF へアップロード（任意）
                                                                  └ LIBERO-Plus で動かして動画を録る
```

**上から順に実行する。書き換えるのは「1. 設定」のセルだけ。**

既定では、公開データ `lerobot/libero_plus` の 40 タスクから、タスクごとに 12 本（長さ順に均等な間隔）を選んで学習する。
teleop のデータセットができたら `DATASET_REPO` を差し替え、`DATASET_REVISION` と `EPISODES_PER_TASK` を `None` にする。

## 用意するもの

| | |
| :--- | :--- |
| GPU | ランタイム → ランタイムのタイプを変更 → GPU。L4 / A100（VRAM 24 GB 以上で bf16 が使えるもの）。π0.5 は約 3B パラメータあり、T4 では足りない見込み |
| Hugging Face token | 左の鍵アイコン（Secrets）に `HF_TOKEN` として登録する。π0.5 の tokenizer は gated の [`google/paligemma-3b-pt-224`](https://huggingface.co/google/paligemma-3b-pt-224) から取るので、先にそのページで利用条件に同意しておく。アップロードするなら書き込み権限つき |
| Weights & Biases | 任意。使うなら `USE_WANDB = True` にして、Secrets に `WANDB_API_KEY` を登録する |

## SmolVLA 版との違い

| | SmolVLA 版 | この notebook（π0.5） |
| :--- | :--- | :--- |
| 初期重み | `lerobot/smolvla_libero_plus` | `lerobot/pi05_libero_finetuned_v044` |
| 学習する部分 | Action Expert（LoRA かフル） | Action Expert に LoRA（視覚エンコーダは凍結） |
| 画像のキー | 入出力の形をデータセットから作り直す | 初期重みの形のまま、`front` → `image`、`wrist` → `image2` と付け替える |
| 正規化の統計量 | 学習データのもの | 既定は初期重みのもの（`PROCESSOR_FROM`） |
| 学習条件の既定 | 2,000 step・batch 8・lr 3e-4・LoRA α 16 | 1,000 step・batch 8・lr 1e-4・LoRA α 8 |
| 精度 | bf16（無ければ fp16） | bf16（無ければ float32） |
| 1 回の推論で実行する action | 10 | 16 |

## データセットの条件

`lerobot/libero_plus` と同じ形式にする。teleop の録画から作るときもこれに合わせる。

| key | 型 | 内容 |
| :--- | :--- | :--- |
| `observation.images.front` | video 256×256×3 | `agentview` カメラ。**env の生画像を 180° 回転**（上下・左右とも反転）したもの |
| `observation.images.wrist` | video 256×256×3 | `robot0_eye_in_hand` カメラ。同じく 180° 回転 |
| `observation.state` | float32 (8,) | eef 位置 3 + eef 姿勢（クォータニオン → 軸角）3 + グリッパ指の位置 2 |
| `action` | float32 (7,) | OSC_POSE の差分 6 + グリッパ 1（teleop の `actions` そのまま） |
| `task` | str | 指示文（teleop の録画 JSON の `language`） |

fps は 20（LIBERO の control_freq と同じ）。

## 1. 設定

In [ ]:
from pathlib import Path

# ---------------------------------------------------------------- データ
# teleop のデータセットを HF に上げたら、その repo に差し替える（revision は None で最新）
DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = "f3f49f426d75030177b18778374005bc12ccd588"
# 学習に使う episode。None なら下の絞り込みで選ぶ
DATASET_EPISODES = None
# 指示文に含まれる文字列で episode を絞る。None なら全タスク
DATASET_TASK_FILTER = None
# タスクごとに、長さ順に均等な間隔で選ぶ本数。None なら絞り込んだ episode を全部使う
# 既定の 12 は lerobot/libero_plus の 40 タスク × 12 本 = 480 episode（parc2026 で使った選び方）
EPISODES_PER_TASK = 12

# データセットの画像キー -> π0.5 の画像キー
RENAME_MAP = {
    "observation.images.front": "observation.images.image",
    "observation.images.wrist": "observation.images.image2",
}

# ---------------------------------------------------------------- 初期重み
BASE_MODEL_REPO = "lerobot/pi05_libero_finetuned_v044"
BASE_MODEL_REVISION = "8e174154ef5f6c60a8da12ae99c303d8963138c1"

# ---------------------------------------------------------------- 学習
# Action Expert に LoRA を付けて学習し、最後に元の重みへマージする（視覚エンコーダは凍結）
STEPS = 1000
BATCH_SIZE = 8
SAVE_FREQ = 500           # この step ごとに checkpoint を保存する
LOG_FREQ = 50
SEED = 42
LORA_R = 16
LORA_ALPHA = 8
LEARNING_RATE = 1e-4
FINAL_LEARNING_RATE = 1e-5
WARMUP_STEPS = 100
# 学習中に画像を一度この解像度へ落としてから 224 へ戻す。None なら素通し（256 → 224）
# 評価で env を低い解像度で描画するときに、その値にして CAMERA_PX もそろえる（7. を参照）
TRAIN_IMAGE_PX = None
TRAIN_IMAGE_RESIZE = "nearest"    # 落とし方。"nearest"（点サンプリング）か "bilinear"
# マージ後のモデルに置く正規化の統計量。"base": 初期重みのもの / "checkpoint": 学習データから計算し直したもの
PROCESSOR_FROM = "base"
USE_WANDB = False
WANDB_PROJECT = "pi05-libero-finetune"
RUN_NAME = f"pi05_lora_s{STEPS}_b{BATCH_SIZE}" + (f"_img{TRAIN_IMAGE_PX}_{TRAIN_IMAGE_RESIZE}" if TRAIN_IMAGE_PX else "")

# ---------------------------------------------------------------- 保存先
WORK_DIR = Path("/content/work")
OUTPUT_DIR = WORK_DIR / "outputs" / RUN_NAME      # lerobot-train の出力（checkpoint）
FINAL_MODEL_DIR = WORK_DIR / "models" / RUN_NAME  # 推論にそのまま使える形のモデル
PUSH_TO_HUB = False
HF_MODEL_REPO = "your-hf-name/pi05-libero-finetuned"   # PUSH_TO_HUB = True のとき
HF_MODEL_PRIVATE = True

# ---------------------------------------------------------------- 評価（動画）
# task id は LIBERO-Plus の suite 内の番号（teleop の選択画面・録画 JSON の task_id と同じ）
EVAL_SUITE = "libero_spatial"
EVAL_TASK_IDS = [0]
EVAL_INIT_INDICES = [0, 1, 2]
EVAL_MAX_STEPS = None     # None なら suite の既定（spatial 220 / object 280 / goal 300 / 10 520）
N_ACTION_STEPS = 16       # 1 回の推論で出す 50 step 分の action のうち、実行する数
EVAL_BASELINE = True      # 追加学習前のモデルも同じ条件で動かして比べる
CAMERA_PX = 256           # env の描画解像度。TRAIN_IMAGE_PX を使うならその値にする
VIDEO_DIR = WORK_DIR / "videos" / RUN_NAME

# ---------------------------------------------------------------- 版の固定
LEROBOT_TAG = "v0.6.0"
LIBERO_PLUS_GIT = "https://github.com/sylvestf/LIBERO-plus.git"
LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_ASSETS_REPO = "Sylvest/LIBERO-plus"   # assets.zip（約 6.4 GB、展開後 約 9 GB）

assert TRAIN_IMAGE_RESIZE in ("nearest", "bilinear"), TRAIN_IMAGE_RESIZE
assert PROCESSOR_FROM in ("base", "checkpoint"), PROCESSOR_FROM
if TRAIN_IMAGE_PX and CAMERA_PX != TRAIN_IMAGE_PX:
    print(f"WARNING: TRAIN_IMAGE_PX={TRAIN_IMAGE_PX} と CAMERA_PX={CAMERA_PX} が違います（学習と評価で画像の見え方がずれる）")
print(f"run: {RUN_NAME}")

## 2. ランタイムを確認する

In [ ]:
import os
import subprocess
import sys

import torch

if sys.version_info < (3, 12):
    raise RuntimeError(f"LeRobot {LEROBOT_TAG} は Python 3.12 以上が必要です（今は {sys.version.split()[0]}）")
if not torch.cuda.is_available():
    raise RuntimeError("GPU ランタイムを選んでください（ランタイム → ランタイムのタイプを変更）")

os.environ.update({
    "HF_HOME": "/content/hf_cache",
    "HF_LEROBOT_HOME": "/content/lerobot_cache",
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    # transformers は TensorFlow / JAX が入っていると読み込み、それらが GPU を掴む。
    # すると MuJoCo が EGL の描画先を作れず、評価が落ちる。π0.5 にはどちらも要らない
    "USE_TF": "0",
    "USE_FLAX": "0",
})
WORK_DIR.mkdir(parents=True, exist_ok=True)

BF16 = torch.cuda.is_bf16_supported()
# π0.5 は bfloat16 か float32 で動かす（fp16 は無い）。bf16 が無い GPU では float32・混合精度なしになる
POLICY_DTYPE = "bfloat16" if BF16 else "float32"
MIXED_PRECISION = "bf16" if BF16 else "no"
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"Python {sys.version.split()[0]}  torch {torch.__version__}  CUDA {torch.version.cuda}")
print(f"GPU    {torch.cuda.get_device_name(0)}  {GPU_GB:.0f} GB  bf16={BF16} -> {POLICY_DTYPE}")
if GPU_GB < 22 or not BF16:
    print("WARNING: π0.5 の学習には VRAM 24 GB 以上で bf16 が使える GPU（L4 / A100）を使ってください")
print(subprocess.run(["df", "-h", "/content"], capture_output=True, text=True).stdout)

## 3. LeRobot を入れる

`lerobot-train` と π0.5・PEFT（LoRA）を使うので `training,pi,peft` を付けて入れる。
parc2026 で π0.5 を学習したときの手順に合わせて、PEFT と Accelerate の版をそろえ、torchao は外す。

In [ ]:
import importlib
import importlib.metadata
import shutil
from pathlib import Path


def run(command, check=True, **kwargs):
    """コマンドを実行し、失敗したら出力の末尾を出して止める。"""
    result = subprocess.run(command, text=True, capture_output=True, **kwargs)
    if check and result.returncode != 0:
        raise RuntimeError(f"{' '.join(map(str, command))}\n{(result.stdout + result.stderr)[-4000:]}")
    return result


LEROBOT_DIR = Path("/content/lerobot")
LEROBOT_SRC = LEROBOT_DIR / "src"

run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "ffmpeg", "libegl1", "libgl1", "libglib2.0-0", "libmagickwand-dev"])

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)
run(["git", "clone", "--quiet", "--depth", "1", "--branch", LEROBOT_TAG,
     "https://github.com/huggingface/lerobot.git", str(LEROBOT_DIR)])
run([sys.executable, "-m", "pip", "install", "--quiet", "-e", f"{LEROBOT_DIR}[training,pi,peft]"])
run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps",
     "peft>=0.18.0,<1.0.0", "accelerate>=1.7.0,<2.0.0"])
if USE_WANDB:
    run([sys.executable, "-m", "pip", "install", "--quiet", "wandb"])
run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

# editable install は次の kernel 起動まで sys.path に載らないので、ここで足す
if str(LEROBOT_SRC) not in sys.path:
    sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

import lerobot  # noqa: E402
import peft  # noqa: E402
import transformers  # noqa: E402
from lerobot.policies.pi05 import modeling_pi05  # noqa: E402

# modeling_pi05 は transformers から PaliGemma を読み込めないと None を入れたまま黙って進む
if modeling_pi05.PaliGemmaForConditionalGenerationWithPiGemma is None:
    raise RuntimeError(f"transformers {transformers.__version__} から PaliGemma を読み込めません")

print(f"LeRobot      {importlib.metadata.version('lerobot')}  ({Path(lerobot.__file__).parent})")
print(f"transformers {transformers.__version__}")
print(f"PEFT         {peft.__version__}")
# LeRobot の依存に合わせて pip が torch を入れ替えると、読み込み済みの torch と食い違う
if importlib.metadata.version("torch").split("+")[0] != torch.__version__.split("+")[0]:
    print(f"WARNING: torch が {torch.__version__} から {importlib.metadata.version('torch')} に入れ替わりました。"
          "ランタイム → セッションを再起動 してから、1. 設定 から順に実行し直してください"
          "（このセルをもう一度実行しても問題ない）")

## 4. Hugging Face にログインする

token は Colab の Secrets（`HF_TOKEN`）から読む。無ければ入力欄が出る（画面には表示されない）。
π0.5 の tokenizer を取る `google/paligemma-3b-pt-224` は gated なので、この token で読めるかもここで確かめる。

In [ ]:
from getpass import getpass

from huggingface_hub import hf_hub_download, login, whoami

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face token: ").strip()
login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"Hugging Face: {whoami(token=HF_TOKEN)['name']}")

try:
    hf_hub_download("google/paligemma-3b-pt-224", "tokenizer_config.json", token=HF_TOKEN)
except Exception as error:
    raise RuntimeError("google/paligemma-3b-pt-224 を読めません。Hugging Face のそのページで"
                       "利用条件に同意してから、このセルを実行し直してください") from error
print("google/paligemma-3b-pt-224: OK")

if USE_WANDB:
    import wandb
    if not os.environ.get("WANDB_API_KEY"):
        try:
            from google.colab import userdata
            os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
        except Exception:
            pass
    if not wandb.login():
        raise RuntimeError("W&B にログインできません。WANDB_API_KEY を確認してください")

## 5. データセットを確かめる

メタ情報だけを読み、形式が π0.5 × LIBERO-Plus の前提に合っているかと、学習に使う episode を確かめる。

`EPISODES_PER_TASK` を指定すると、タスクごとに episode を長さ順に並べ、最短から最長まで均等な間隔で選ぶ。
短いものだけ・長いものだけに偏らせないための選び方。

In [ ]:
from collections import Counter

import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata(DATASET_REPO, revision=DATASET_REVISION)
REQUIRED = {
    "observation.images.front": (256, 256, 3),
    "observation.images.wrist": (256, 256, 3),
    "observation.state": (8,),
    "action": (7,),
}
problems = [f"{key}: {tuple(meta.features[key]['shape']) if key in meta.features else '無い'} (期待 {shape})"
            for key, shape in REQUIRED.items()
            if key not in meta.features or tuple(meta.features[key]["shape"]) != shape]
if problems:
    raise RuntimeError("データセットの形式が前提と違います:\n  " + "\n  ".join(problems))
if set(RENAME_MAP) - set(meta.features):
    raise RuntimeError(f"RENAME_MAP のキーがデータセットにありません: {sorted(set(RENAME_MAP) - set(meta.features))}")
if meta.fps != 20:
    print(f"WARNING: fps={meta.fps}（LIBERO の control_freq は 20）")

episodes = (meta.episodes.select_columns(["episode_index", "tasks", "length"]).to_pandas()
            .sort_values("episode_index", kind="stable"))
episodes["task"] = episodes["tasks"].apply(lambda tasks: tasks[0])


def evenly_by_length(group, count):
    """長さ順に並べ、最短から最長まで均等な間隔で count 本を選ぶ。"""
    ordered = group.sort_values("length", kind="stable")
    count = min(count, len(ordered))
    if count == 1:
        return ordered.head(1)
    return ordered.iloc[[round(i * (len(ordered) - 1) / (count - 1)) for i in range(count)]]


if DATASET_EPISODES is not None:
    selected = episodes[episodes["episode_index"].isin(DATASET_EPISODES)]
else:
    selected = episodes
    if DATASET_TASK_FILTER:
        selected = selected[selected["tasks"].apply(lambda tasks: any(DATASET_TASK_FILTER in t for t in tasks))]
    if EPISODES_PER_TASK and not selected.empty:
        selected = pd.concat([evenly_by_length(group, EPISODES_PER_TASK)
                              for _, group in selected.groupby("task", sort=True)])
    selected = selected.sort_values("episode_index")
if selected.empty:
    raise RuntimeError("学習に使う episode がありません。DATASET_EPISODES / DATASET_TASK_FILTER を確認してください")

# 全 episode を使うときは --dataset.episodes を渡さない（コマンドが長くなりすぎないように）
TRAIN_EPISODES = None if len(selected) == len(episodes) else [int(i) for i in selected["episode_index"]]
train_frames = int(selected["length"].sum())

print(f"dataset   {DATASET_REPO}@{DATASET_REVISION or 'main'}")
print(f"          {meta.total_episodes:,} episodes / {meta.total_frames:,} frames / fps {meta.fps}")
print(f"学習に使う {len(selected):,} episodes / {selected['task'].nunique()} タスク / {train_frames:,} frames")
for task, n in Counter(selected["task"]).most_common(10):
    print(f"  {n:4d}  {task}")
print(f"学習で見るサンプル数 {STEPS * BATCH_SIZE:,}（各フレーム平均 {STEPS * BATCH_SIZE / train_frames:.2f} 回）")

## 6. 初期重みを取る

In [ ]:
import json

from huggingface_hub import snapshot_download

BASE_MODEL_DIR = Path(snapshot_download(
    BASE_MODEL_REPO,
    revision=BASE_MODEL_REVISION,
    allow_patterns=["config.json", "model.safetensors", "train_config.json", "policy_*"],
))
assert (BASE_MODEL_DIR / "model.safetensors").is_file(), BASE_MODEL_DIR
base_config = json.loads((BASE_MODEL_DIR / "config.json").read_text(encoding="utf-8"))
missing = set(RENAME_MAP.values()) - set(base_config["input_features"])
if missing:
    raise RuntimeError(f"初期重みの入力に {sorted(missing)} がありません（RENAME_MAP を確認）: "
                       f"{sorted(base_config['input_features'])}")
print(f"base model {BASE_MODEL_REPO}@{BASE_MODEL_REVISION}")
print(f"           {BASE_MODEL_DIR}")
print(f"           dtype={base_config['dtype']}  inputs={sorted(base_config['input_features'])}")

## 7. 追加学習する

`lerobot-train` を別プロセスで動かし、`LOG_FREQ` ごとに loss と learning rate を表示する。

- 入出力の形は初期重みのものを使い、データセットの画像キーを `RENAME_MAP` で π0.5 のキーへ付け替える
- 視覚エンコーダは凍結し、行動を出す Action Expert だけに LoRA を付けて学習する（`train_expert_only`）
- `SAVE_FREQ` ごとに checkpoint が `OUTPUT_DIR/checkpoints/<step>/` に残る
- 最初の 1 回はデータセットの動画のダウンロードと読み込みで時間がかかる

### 低い解像度を経由して学習する（`TRAIN_IMAGE_PX`、任意）

π0.5 は画像を 224px にして読む。評価で env を低い解像度（例 128px）で描画すると、
学習時（256 → 224 の縮小）と評価時（128 → 224 の拡大）で見え方が変わる。
`TRAIN_IMAGE_PX` を指定すると、学習でも画像を一度その解像度へ落としてから 224 へ戻し、見え方をそろえる。

`lerobot-train` は別プロセスで動くので、notebook の中で関数を差し替えても効かない。
そこで LeRobot の `modeling_pi05.py` を直接書き換える。書き換えた部分は環境変数 `PI05_TRAIN_IMAGE_PX` が
無ければ何もしないので、評価（11.）には影響しない。落とし方（`TRAIN_IMAGE_RESIZE`）を変えるときは、
3. から実行し直して LeRobot を取り直す。

In [ ]:
modeling_path = LEROBOT_SRC / "lerobot/policies/pi05/modeling_pi05.py"
MARKER = "TRAIN_IMAGE_PX patch"
source = modeling_path.read_text(encoding="utf-8")

if not TRAIN_IMAGE_PX:
    print("TRAIN_IMAGE_PX = None なので書き換えない（256 → 224 のまま学習する）")
elif MARKER in source:
    if f'mode="{TRAIN_IMAGE_RESIZE}"' not in source:
        raise RuntimeError("別の TRAIN_IMAGE_RESIZE で書き換え済みです。3. から実行し直して LeRobot を取り直してください")
    print("書き換え済み")
else:
    anchor_import = "from lerobot.utils.import_utils import _transformers_available, require_package\n"
    anchor_resize = (
        "            # from openpi preprocess_observation_pytorch: Resize with padding if needed\n"
        "            if img.shape[1:3] != self.config.image_resolution:\n"
    )
    for anchor in (anchor_import, anchor_resize):
        if source.count(anchor) != 1:
            raise RuntimeError(f"modeling_pi05.py に書き換える位置が見つかりません。LeRobot の版を確認してください: {anchor!r}")
    source = source.replace(anchor_import, anchor_import + (
        f"\n# {MARKER}: 学習中だけ、画像を一度 PI05_TRAIN_IMAGE_PX へ落としてから 224 へ戻す（0 なら素通し）\n"
        "import os as _os\n"
        '_TRAIN_IMAGE_PX = int(_os.environ.get("PI05_TRAIN_IMAGE_PX", "0"))\n'
    ), 1)
    # ここでの img は [B, H, W, C]（resize_with_pad_torch の想定と同じ）
    source = source.replace(anchor_resize, (
        f"            # {MARKER}\n"
        "            if _TRAIN_IMAGE_PX and img.shape[1:3] != (_TRAIN_IMAGE_PX, _TRAIN_IMAGE_PX):\n"
        f'                img = resize_with_pad_torch(img, _TRAIN_IMAGE_PX, _TRAIN_IMAGE_PX, mode="{TRAIN_IMAGE_RESIZE}")\n'
    ) + anchor_resize, 1)
    modeling_path.write_text(source, encoding="utf-8")
    patched = modeling_path.read_text(encoding="utf-8")
    start = patched.index(f"            # {MARKER}")
    print(f"書き換えた: {modeling_path}")
    print(patched[start:patched.index(anchor_resize, start)])

In [ ]:
import re
from collections import deque

command = [
    "lerobot-train",
    f"--policy.path={BASE_MODEL_DIR}",
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    "--policy.device=cuda",
    f"--policy.dtype={POLICY_DTYPE}",
    # 初期重みの config は compile_model=true。学習中の torch.compile は
    # flash attention の backward で落ちるので切る
    "--policy.compile_model=false",
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    "--policy.empty_cameras=0",
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    f"--dataset.repo_id={DATASET_REPO}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    "--rename_map=" + json.dumps(RENAME_MAP, separators=(",", ":")),
    f"--output_dir={OUTPUT_DIR}",
    f"--job_name={RUN_NAME}",
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=0",
    "--persistent_workers=false",
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    "--save_checkpoint=true",
    f"--save_freq={SAVE_FREQ}",
    f"--log_freq={LOG_FREQ}",
    f"--wandb.enable={str(USE_WANDB).lower()}",
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
    "--save_checkpoint_to_hub=false",
]
if USE_WANDB:
    command.append(f"--wandb.project={WANDB_PROJECT}")
if DATASET_REVISION:
    command.append(f"--dataset.revision={DATASET_REVISION}")
if TRAIN_EPISODES is not None:
    command.append("--dataset.episodes=" + json.dumps(TRAIN_EPISODES, separators=(",", ":")))

env = dict(os.environ,
           PYTHONPATH=f"{LEROBOT_SRC}{os.pathsep}{os.environ.get('PYTHONPATH', '')}",
           ACCELERATE_MIXED_PRECISION=MIXED_PRECISION,
           PI05_TRAIN_IMAGE_PX=str(TRAIN_IMAGE_PX or 0),
           PYTHONUNBUFFERED="1",
           TQDM_DISABLE="1")
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)

print(f"LoRA r={LORA_R} alpha={LORA_ALPHA} / {STEPS} steps / batch {BATCH_SIZE} / "
      f"lr {LEARNING_RATE:g} -> {FINAL_LEARNING_RATE:g} / image {TRAIN_IMAGE_PX or 256}px")
process = subprocess.Popen(command, cwd=LEROBOT_DIR, env=env, text=True, bufsize=1,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
recent = deque(maxlen=60)
losses = []
for line in process.stdout:
    line = line.strip()
    if not line:
        continue
    recent.append(line)
    step = re.search(r"step:\s*([0-9.]+[KMB]?)", line)
    loss = re.search(r"loss:\s*([0-9.eE+-]+)", line)
    if step and loss:
        lr = re.search(r"lr:\s*([0-9.eE+-]+)", line)
        losses.append(float(loss.group(1)))
        print(f"step {step.group(1):>6}  loss {loss.group(1)}  lr {lr.group(1) if lr else '-'}", flush=True)
if process.wait() != 0:
    print("\n".join(recent))
    raise RuntimeError(f"学習が失敗しました（exit {process.returncode}）")
print("学習が終わりました")

## 8. 学習済みモデルをまとめる

LoRA は差分だけが保存されるので、元の重みへマージして普通の π0.5 モデルにする（CPU で行う）。
前処理・後処理（正規化など）の設定も一緒に置く。

正規化の統計量は、既定（`PROCESSOR_FROM = "base"`）では初期重みのものを使う。
`lerobot-train` が checkpoint に書く統計量は学習データから計算し直したもので、初期重みのものと少しずれる。
parc2026 では、学習の前後を比べたときに変わるのが重みだけになるよう、初期重みの統計量を使った。
学習データの分布が初期重みのものと大きく違うなら `"checkpoint"` にする。

In [ ]:
import gc

from lerobot.configs import PreTrainedConfig
from lerobot.policies.pi05.modeling_pi05 import PI05Policy
from peft import PeftModel
from safetensors import safe_open

checkpoint_dir = OUTPUT_DIR / "checkpoints" / "last" / "pretrained_model"
if not checkpoint_dir.is_dir():
    checkpoint_dir = OUTPUT_DIR / "checkpoints" / f"{STEPS:06d}" / "pretrained_model"
print(f"checkpoint {checkpoint_dir}")

config = PreTrainedConfig.from_pretrained(checkpoint_dir)
config.device = "cpu"
config.pretrained_path = BASE_MODEL_DIR
config.use_peft = False
base = PI05Policy.from_pretrained(BASE_MODEL_DIR, config=config, strict=False)
merged = PeftModel.from_pretrained(base, checkpoint_dir, is_trainable=False,
                                   torch_device="cpu").merge_and_unload(safe_merge=True)
merged.config.use_peft = False
merged.config.compile_model = False
merged.config.pretrained_path = None
merged.config.push_to_hub = False
merged.config.repo_id = None
merged.config.device = None

shutil.rmtree(FINAL_MODEL_DIR, ignore_errors=True)
merged.save_pretrained(FINAL_MODEL_DIR)
processor_dir = BASE_MODEL_DIR if PROCESSOR_FROM == "base" else checkpoint_dir
for pattern in ("policy_preprocessor*", "policy_postprocessor*"):
    found = list(processor_dir.glob(pattern))
    if not found:
        raise FileNotFoundError(f"{processor_dir} に {pattern} がありません")
    for path in found:
        shutil.copy2(path, FINAL_MODEL_DIR / path.name)
del base, merged
gc.collect()
torch.cuda.empty_cache()

with safe_open(FINAL_MODEL_DIR / "model.safetensors", framework="pt") as weights:
    leftover = [k for k in weights.keys() if "lora_" in k.lower()]
if leftover:
    raise RuntimeError(f"LoRA の重みが残っています: {leftover[:3]}")
for name in sorted(p.name for p in FINAL_MODEL_DIR.iterdir()):
    print(f"  {name}")
print(f"モデル {FINAL_MODEL_DIR}（processor: {PROCESSOR_FROM}）")

## 9. Hugging Face へアップロードする（任意）

`PUSH_TO_HUB = True` のときだけ動く。既定では private の model repo に置く。

In [ ]:
from huggingface_hub import HfApi

if PUSH_TO_HUB:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_MODEL_REPO, repo_type="model", private=HF_MODEL_PRIVATE, exist_ok=True)
    commit = api.upload_folder(
        folder_path=str(FINAL_MODEL_DIR), repo_id=HF_MODEL_REPO, repo_type="model",
        commit_message=(f"pi05 LoRA fine-tune: {DATASET_REPO}, {STEPS} steps, batch {BATCH_SIZE}, "
                        f"lr {LEARNING_RATE:g}, r {LORA_R}, alpha {LORA_ALPHA}, "
                        f"image {TRAIN_IMAGE_PX or 256}px, processor {PROCESSOR_FROM} (LeRobot {LEROBOT_TAG})"),
    )
    print(f"https://huggingface.co/{HF_MODEL_REPO}  revision {commit.oid}")
else:
    print("PUSH_TO_HUB = False なのでアップロードしません")

## 10. LIBERO-Plus を準備する

シミュレータ（robosuite / MuJoCo）と LIBERO-Plus 本体、物体やテクスチャの assets を入れる。
assets は約 6.4 GB あり、ダウンロードと展開に数分かかる（2 回目以降は飛ばす）。

Colab では `MUJOCO_GL=egl` だけだと **描画が CPU（llvmpipe）に落ちる**ことがあるので、
NVIDIA の EGL を明示する設定ファイルを作る。実際の描画先は次のセルの最初に `[render]` として表示される。

In [ ]:
import zipfile

from huggingface_hub import hf_hub_download

LIBERO_PLUS_DIR = Path("/content/LIBERO-plus")
LIBERO_ROOT = LIBERO_PLUS_DIR / "libero" / "libero"
ASSETS_DIR = LIBERO_ROOT / "assets"

run([sys.executable, "-m", "pip", "install", "--quiet",
     "robosuite==1.4.1", "mujoco==3.7.0", "bddl==1.0.1", "easydict==1.13", "future",
     "Wand==0.6.13", "scikit-image", "gym==0.26.2", "matplotlib", "PyOpenGL", "imageio[ffmpeg]"])

if not LIBERO_PLUS_DIR.is_dir():
    run(["git", "clone", "--quiet", LIBERO_PLUS_GIT, str(LIBERO_PLUS_DIR)])
run(["git", "-C", str(LIBERO_PLUS_DIR), "checkout", "--quiet", LIBERO_PLUS_SHA])

# LIBERO は初回 import 時にパスを対話で聞くので、先に設定ファイルを書いておく
libero_config = Path.home() / ".libero" / "config.yaml"
libero_config.parent.mkdir(parents=True, exist_ok=True)
libero_config.write_text("".join(f"{key}: {value}\n" for key, value in {
    "benchmark_root": LIBERO_ROOT,
    "bddl_files": LIBERO_ROOT / "bddl_files",
    "init_states": LIBERO_ROOT / "init_files",
    "datasets": LIBERO_ROOT / "datasets",
    "assets": ASSETS_DIR,
}.items()))

if not (ASSETS_DIR / "scenes").is_dir():
    archive = Path(hf_hub_download(LIBERO_PLUS_ASSETS_REPO, "assets.zip", repo_type="dataset",
                                   local_dir="/content/libero_plus_assets"))
    extract_dir = Path("/content/libero_plus_assets/extracted")
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(extract_dir)
    # zip の中で assets/ がどの深さにあっても拾えるように、scenes/ を持つディレクトリを探す
    found = next(p.parent for p in sorted(extract_dir.rglob("scenes"), key=lambda p: len(p.parts)) if p.is_dir())
    shutil.rmtree(ASSETS_DIR, ignore_errors=True)
    shutil.move(str(found), str(ASSETS_DIR))
    archive.unlink()
    shutil.rmtree(extract_dir, ignore_errors=True)
missing = [name for name in ("scenes", "textures", "stable_scanned_objects", "articulated_objects")
           if not (ASSETS_DIR / name).is_dir()]
if missing:
    raise RuntimeError(f"assets が足りません: {missing}")

# NVIDIA の EGL を使わせる（Colab の glvnd には Mesa しか登録されていないことがある）
NVIDIA_LIB = Path("/usr/lib64-nvidia")
egl_icd = WORK_DIR / "10_nvidia_egl.json"
egl_icd.write_text(json.dumps({"file_format_version": "1.0.0",
                               "ICD": {"library_path": str(NVIDIA_LIB / "libEGL_nvidia.so.0")}}))
SIM_ENV = dict(os.environ,
               MUJOCO_GL="egl",
               PYOPENGL_PLATFORM="egl",
               PYTHONPATH=f"{LIBERO_PLUS_DIR}{os.pathsep}{LEROBOT_SRC}",
               PYTHONUNBUFFERED="1")
if (NVIDIA_LIB / "libEGL_nvidia.so.0").exists():
    SIM_ENV["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(egl_icd)
    SIM_ENV["LD_LIBRARY_PATH"] = f"{NVIDIA_LIB}{os.pathsep}{os.environ.get('LD_LIBRARY_PATH', '')}"
print(f"LIBERO-Plus {LIBERO_PLUS_SHA}  assets {ASSETS_DIR}")

## 11. LIBERO-Plus で動かして動画を録る

学習済みモデル（`EVAL_BASELINE = True` なら追加学習前のモデルも）を、同じタスク・同じ初期配置で動かす。
1 回ごとに mp4（左: 正面カメラ、右: 手首カメラ）と結果の 1 行（`results.jsonl`）を残す。

観測は学習データと同じ形にそろえる：画像は 180° 回転、状態は 8 次元、指示文は LIBERO-Plus のタスクの文。
画像は π0.5 のキー（`observation.images.image` / `image2`）で渡す。成功はゴール条件（BDDL の `:goal`）の成立で判定する。

In [ ]:
%%writefile /content/work/rollout_pi05_libero_plus.py
"""学習した π0.5 を LIBERO-Plus で動かし、成否と動画を残す。

ノートブックから別プロセスで起動する。MuJoCo の描画設定（EGL）を import より前に
入れるためと、GPU メモリを学習側と分けるため。

観測は学習データ（lerobot/libero_plus と同じ形式）にそろえる。

- 画像: env の生画像を 180° 回転（上下と左右を両方反転）。データセットはこの向きで保存されている
- 状態: [eef 位置 3, eef 姿勢の軸角 3, グリッパ指の位置 2] の 8 次元
- 画像のキー名: π0.5 の入力名（既定 observation.images.image / image2）で渡す。
  正規化と指示文の tokenize は、モデルに同梱の preprocessor が行う
"""

import argparse
import json
import os
import sys
import time
from pathlib import Path

import numpy as np

# suite ごとの既定の最大 step 数（LeRobot の LIBERO 評価と同じ値）
MAX_STEPS = {"libero_spatial": 220, "libero_object": 280, "libero_goal": 300,
             "libero_10": 520, "libero_90": 400}
# 初期配置の直後は物体が落ち着くまで待つ（グリッパを開いたまま何もしない）
NUM_WAIT_STEPS = 10
DUMMY_ACTION = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, -1.0]


def parse_args():
    p = argparse.ArgumentParser(description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--model", required=True, help="π0.5 のディレクトリ（model.safetensors と processor を含む）")
    p.add_argument("--label", default="model", help="動画名と結果に付ける名前")
    p.add_argument("--suite", default="libero_spatial", choices=sorted(MAX_STEPS))
    p.add_argument("--task-ids", type=int, nargs="+", default=[0])
    p.add_argument("--init-indices", type=int, nargs="+", default=[0])
    p.add_argument("--max-steps", type=int, default=None, help="省略時は suite ごとの既定値")
    p.add_argument("--n-action-steps", type=int, default=16,
                   help="1 回の推論で出した action 列のうち、実際に実行する数")
    p.add_argument("--camera-px", type=int, default=256, help="env の描画解像度")
    p.add_argument("--front-key", default="observation.images.image")
    p.add_argument("--wrist-key", default="observation.images.image2")
    p.add_argument("--dtype", default=None, choices=["bfloat16", "float32"],
                   help="省略時はモデルの config のまま")
    p.add_argument("--seed", type=int, default=7)
    p.add_argument("--device", default=None, help="省略時は cuda → mps → cpu の順で選ぶ")
    p.add_argument("--fps", type=int, default=20)
    p.add_argument("--out", type=Path, required=True)
    return p.parse_args()


def probe_renderer():
    """MuJoCo が実際にどの GPU で描画しているかを返す（Linux のみ）。

    Colab では MUJOCO_GL=egl でも CPU 描画（Mesa / llvmpipe）に落ちることがある。
    落ちても動くが遅いので、警告だけ出して続ける。
    """
    if sys.platform != "linux":
        return None
    import mujoco
    from OpenGL import GL

    model = mujoco.MjModel.from_xml_string("<mujoco><worldbody><geom size='.1'/></worldbody></mujoco>")
    data = mujoco.MjData(model)
    mujoco.mj_forward(model, data)
    renderer = mujoco.Renderer(model, 8, 8)
    renderer.update_scene(data)
    renderer.render()
    vendor = GL.glGetString(GL.GL_VENDOR).decode()
    name = GL.glGetString(GL.GL_RENDERER).decode()
    renderer.close()
    return f"{vendor} / {name}"


def patch_torch_load():
    """LIBERO の初期状態ファイルは weights_only=False でないと読めない（torch 2.6 以降の既定は True）。"""
    import torch

    original = torch.load

    def load(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return original(*args, **kwargs)

    torch.load = load


def state8(obs):
    from robosuite.utils.transform_utils import quat2axisangle

    return np.concatenate([
        np.asarray(obs["robot0_eef_pos"], dtype=np.float64),
        quat2axisangle(np.asarray(obs["robot0_eef_quat"], dtype=np.float64)),
        np.asarray(obs["robot0_gripper_qpos"], dtype=np.float64),
    ]).astype(np.float32)


def upright(image):
    """env の生画像（上下逆さま）を 180° 回して、データセットと同じ向きにする。"""
    return np.flip(np.asarray(image), axis=(0, 1)).copy()


def to_model_input(obs, args, instruction):
    import torch

    def chw(image):
        return torch.from_numpy(upright(image)).permute(2, 0, 1).float().div(255.0)

    return {
        args.front_key: chw(obs["agentview_image"]),
        args.wrist_key: chw(obs["robot0_eye_in_hand_image"]),
        "observation.state": torch.from_numpy(state8(obs)),
        "task": instruction,
    }


def load_policy(args, device):
    from lerobot.configs import PreTrainedConfig
    from lerobot.policies.factory import make_pre_post_processors
    from lerobot.policies.pi05.modeling_pi05 import PI05Policy

    config = PreTrainedConfig.from_pretrained(args.model)
    if not 1 <= args.n_action_steps <= config.chunk_size:
        raise SystemExit(f"--n-action-steps は 1..{config.chunk_size} の範囲で指定する")
    config.n_action_steps = args.n_action_steps
    # 初期重みの config は compile_model=true。推論でコンパイルしても得はないので切る
    config.compile_model = False
    config.device = device
    if args.dtype:
        config.dtype = args.dtype
    policy = PI05Policy.from_pretrained(args.model, config=config)
    policy.to(device)
    policy.eval()
    preprocessor, postprocessor = make_pre_post_processors(
        policy.config,
        pretrained_path=args.model,
        preprocessor_overrides={"device_processor": {"device": device}},
    )
    expected = list(policy.config.image_features)
    print(f"[policy] {args.model}  image keys={expected}  n_action_steps={args.n_action_steps}  "
          f"dtype={policy.config.dtype}", flush=True)
    return policy, preprocessor, postprocessor


def write_video(path, frames, fps):
    import imageio.v2 as imageio

    path.parent.mkdir(parents=True, exist_ok=True)
    with imageio.get_writer(path, fps=fps, codec="libx264", quality=7, macro_block_size=1) as writer:
        for frame in frames:
            writer.append_data(frame)


def run_episode(env, init_state, policy, preprocessor, postprocessor, args, instruction, max_steps):
    import torch

    env.reset()
    obs = env.set_init_state(init_state)
    for _ in range(NUM_WAIT_STEPS):
        obs, _, _, _ = env.step(DUMMY_ACTION)
    policy.reset()

    frames = [np.concatenate([upright(obs["agentview_image"]), upright(obs["robot0_eye_in_hand_image"])], axis=1)]
    success, steps = False, 0
    for steps in range(1, max_steps + 1):
        batch = preprocessor(to_model_input(obs, args, instruction))
        with torch.inference_mode():
            action = policy.select_action(batch)
        action = postprocessor(action)
        action = np.asarray(action.detach().float().cpu(), dtype=np.float64).reshape(-1)
        if action.shape != (7,) or not np.isfinite(action).all():
            raise RuntimeError(f"action が不正: shape={action.shape}")
        obs, _, done, _ = env.step(action.tolist())
        frames.append(np.concatenate([upright(obs["agentview_image"]), upright(obs["robot0_eye_in_hand_image"])], axis=1))
        if done or env.check_success():
            success = True
            break
    return success, steps, frames


def main():
    args = parse_args()
    renderer = probe_renderer()
    if renderer is not None:
        print(f"[render] {renderer}", flush=True)
        if "llvmpipe" in renderer or "Mesa" in renderer:
            print("[render] WARNING: CPU 描画です。動きますが遅くなります（EGL の設定を確認）", flush=True)

    import torch

    patch_torch_load()
    from libero.libero import benchmark
    from libero.libero.envs import OffScreenRenderEnv

    device = args.device or ("cuda" if torch.cuda.is_available()
                             else "mps" if torch.backends.mps.is_available() else "cpu")
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    policy, preprocessor, postprocessor = load_policy(args, device)

    suite = benchmark.get_benchmark_dict()[args.suite](task_order_index=0)
    max_steps = args.max_steps or MAX_STEPS[args.suite]
    args.out.mkdir(parents=True, exist_ok=True)
    results_path = args.out / "results.jsonl"

    results = []
    for task_id in args.task_ids:
        task = suite.get_task(task_id)
        bddl = suite.get_task_bddl_file_path(task_id)
        init_states = suite.get_task_init_states(task_id)
        env = OffScreenRenderEnv(bddl_file_name=bddl, camera_heights=args.camera_px,
                                 camera_widths=args.camera_px, horizon=max_steps + NUM_WAIT_STEPS + 10,
                                 ignore_done=True)
        env.seed(args.seed)
        print(f"[task] {args.suite} #{task_id}  {task.language}", flush=True)
        for init_index in args.init_indices:
            if not 0 <= init_index < len(init_states):
                print(f"  [skip] init {init_index} は範囲外（0..{len(init_states) - 1}）", flush=True)
                continue
            t0 = time.time()
            success, steps, frames = run_episode(env, init_states[init_index], policy, preprocessor,
                                                 postprocessor, args, task.language, max_steps)
            video = args.out / f"{args.label}_{args.suite}_t{task_id}_i{init_index}_{'success' if success else 'fail'}.mp4"
            write_video(video, frames, args.fps)
            row = dict(label=args.label, model=str(args.model), suite=args.suite, task_id=task_id,
                       task=task.name, instruction=task.language, init_index=init_index,
                       success=success, steps=steps, max_steps=max_steps,
                       n_action_steps=args.n_action_steps, camera_px=args.camera_px, seed=args.seed,
                       seconds=round(time.time() - t0, 1), video=str(video))
            results.append(row)
            with results_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
            print(f"  init {init_index}: {'SUCCESS' if success else 'fail'}  steps={steps}  "
                  f"{row['seconds']}s  -> {video.name}", flush=True)
        env.close()

    n_success = sum(r["success"] for r in results)
    print(f"[done] {args.label}: {n_success}/{len(results)} success", flush=True)


if __name__ == "__main__":
    main()

In [ ]:
def rollout(model_dir, label):
    command = [sys.executable, str(WORK_DIR / "rollout_pi05_libero_plus.py"),
               "--model", str(model_dir), "--label", label,
               "--suite", EVAL_SUITE,
               "--task-ids", *map(str, EVAL_TASK_IDS),
               "--init-indices", *map(str, EVAL_INIT_INDICES),
               "--n-action-steps", str(N_ACTION_STEPS),
               "--camera-px", str(CAMERA_PX),
               "--front-key", RENAME_MAP["observation.images.front"],
               "--wrist-key", RENAME_MAP["observation.images.wrist"],
               "--dtype", POLICY_DTYPE,
               "--seed", str(SEED),
               "--out", str(VIDEO_DIR)]
    if EVAL_MAX_STEPS:
        command += ["--max-steps", str(EVAL_MAX_STEPS)]
    process = subprocess.Popen(command, env=SIM_ENV, text=True, bufsize=1,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    recent = deque(maxlen=40)
    for line in process.stdout:
        recent.append(line.rstrip())
        # スクリプト自身の行だけ出す（LIBERO-Plus はタスク番号を数千件並べたログを出す）
        if line.startswith(("[policy]", "[render]", "[task]", "[done]", "  init", "  [skip]")):
            print(line.rstrip(), flush=True)
    if process.wait() != 0:
        print("\n".join(recent))
        raise RuntimeError(f"{label} の rollout が失敗しました（exit {process.returncode}）")


VIDEO_DIR.mkdir(parents=True, exist_ok=True)
(VIDEO_DIR / "results.jsonl").unlink(missing_ok=True)
if EVAL_BASELINE:
    rollout(BASE_MODEL_DIR, "base")
rollout(FINAL_MODEL_DIR, "finetuned")

## 12. 結果と動画を見る

In [ ]:
import pandas as pd
from IPython.display import HTML, Video, display

results = pd.read_json(VIDEO_DIR / "results.jsonl", lines=True)
display(results[["label", "suite", "task_id", "init_index", "success", "steps", "seconds"]])
display(results.groupby("label")["success"].agg(["sum", "count", "mean"]).rename(
    columns={"sum": "成功", "count": "試行", "mean": "成功率"}))

for row in results.itertuples():
    display(HTML(f"<b>{row.label}</b> {row.suite} #{row.task_id} init {row.init_index} — "
                 f"{'SUCCESS' if row.success else 'fail'}（{row.steps} steps）<br>{row.instruction}"))
    display(Video(row.video, embed=True, width=512))

動画・結果・学習済みモデルを手元に持ち帰る（Colab の VM は切断すると消える）。

In [ ]:
archive = shutil.make_archive(str(WORK_DIR / f"{RUN_NAME}_videos"), "zip", VIDEO_DIR)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception:
    pass